## 순차 Lag OOF (검증 = 테스트와 동일한 조건)

- **원본 노트북과의 차이**: `target_lag`를 학습 전에 전체 train에 한 번 넣지 않습니다. 각 fold에서 **학습 구간(tr)** 은 해당 구간의 **실제 타깃**으로만 lag를 만들고, **검증 구간(val)** 은 **이전 타임슬롯의 OOF 예측값**으로 lag를 채운 뒤 슬롯별로 순차 예측합니다(테스트 섹션 9와 동일한 논리).
- **Early stopping**: 학습 속도를 위해 검증 세트는 여전히 **실제 lag**가 들어간 행렬로 두어 iteration을 고릅니다. 리포트되는 **OOF MAE / fold MAE**는 모두 **순차 예측 경로**만 사용합니다.
- 이 OOF는 리더보드(Public)와 같은 **자기회귀 lag** 가정에 가깝습니다.

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import os
import sys
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import time
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')


def _print_progress(msg):
    sys.stdout.write('\r' + msg)
    sys.stdout.flush()


def elapsed(start):
    s = int(time.time() - start)
    return f"{s//60}m {s%60:02d}s"


def section(title):
    print(f"\n{'='*55}\n  {title}\n{'='*55}")


class LGBProgressCallback:
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold = fold
        self.n_folds = n_folds
        self.total = total_rounds
        self.log_every = log_every
        self.start = time.time()
        self.best_mae = float('inf')
        self.best_iter = 0

    def __call__(self, env):
        it = env.iteration + 1
        mae = env.evaluation_result_list[0][2]
        if mae < self.best_mae:
            self.best_mae = mae
            self.best_iter = it
        if it % self.log_every == 0 or it == self.total:
            filled = int(30 * it / self.total)
            bar = '█' * filled + '░' * (30 - filled)
            _print_progress(
                f"  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%"
                f"  iter {it:>5}  val_MAE(진짜 lag, ES용) {mae:.4f}"
                f"  best {self.best_mae:.4f}@{self.best_iter}  {elapsed(self.start)}"
            )


class XGBProgressCallback(xgb.callback.TrainingCallback):
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold = fold
        self.n_folds = n_folds
        self.total = total_rounds
        self.log_every = log_every
        self.start = time.time()
        self.best_mae = float('inf')
        self.best_iter = 0

    def after_iteration(self, model, epoch, evals_log):
        it = epoch + 1
        try:
            mae = list(evals_log.values())[0]['mae'][-1]
        except Exception:
            return False
        if mae < self.best_mae:
            self.best_mae = mae
            self.best_iter = it
        if it % self.log_every == 0:
            filled = int(30 * it / self.total)
            bar = '█' * filled + '░' * (30 - filled)
            _print_progress(
                f"  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%"
                f"  iter {it:>5}  val_MAE(진짜 lag, ES용) {mae:.4f}"
                f"  best {self.best_mae:.4f}@{self.best_iter}  {elapsed(self.start)}"
            )
        return False


def cat_verbose_step(n_estimators):
    return max(1, n_estimators // 5)


def add_target_lags_from_true_y(df):
    """df: 단일 시나리오 집합(한 fold의 train 등). 실제 TARGET으로 lag 생성."""
    out = df.sort_values(['scenario_id', 'ID']).copy()
    for lag in (1, 2, 3):
        col = f'target_lag{lag}'
        out[col] = out.groupby('scenario_id')[TARGET].shift(lag)
        scen_mean = out.groupby('scenario_id')[TARGET].transform('mean')
        out[col] = out[col].fillna(scen_mean)
    return out


def sequential_slot_predictions(model, frame, feature_cols, gm, predict_log):
    """
    테스트 섹션과 동일: timeslot 순서로 lag를 이전 슬롯의 '예측값'으로 채우며 예측.
    frame: 검증 fold 행 (index 보존), timeslot 컬럼 필요.
    predict_log: (model, X_df) -> log1p 공간 예측
    """
    va = frame.copy()
    for lag in (1, 2, 3):
        va[f'target_lag{lag}'] = gm
    pred_buf = pd.Series(np.nan, index=va.index, dtype=np.float64)
    max_slot = int(va['timeslot'].max())
    for slot in range(max_slot + 1):
        slot_mask = va['timeslot'].to_numpy() == slot
        slot_idx = va.index[slot_mask]
        for lag in (1, 2, 3):
            col = f'target_lag{lag}'
            prev_slot = slot - lag
            if prev_slot >= 0:
                prev_mask = va['timeslot'].to_numpy() == prev_slot
                prev_df = va.loc[prev_mask, ['scenario_id']].copy()
                prev_df['pred'] = pred_buf.loc[va.index[prev_mask]].to_numpy()
                scen_to_pred = prev_df.groupby('scenario_id', sort=False)['pred'].last()
                va.loc[slot_idx, col] = (
                    va.loc[slot_idx, 'scenario_id'].map(scen_to_pred).fillna(gm)
                )
            else:
                va.loc[slot_idx, col] = gm
        X_slot = va.loc[slot_idx, feature_cols]
        pl = predict_log(model, X_slot)
        pred_buf.loc[slot_idx] = np.expm1(np.asarray(pl, dtype=np.float64))
    return pred_buf


# ============================================================
# 1. 데이터 로드 (프로젝트 data/ 폴더 자동 탐색)
# ============================================================
def _resolve_data_dir() -> str:
    """저장소 루트의 data/ (train.csv 포함). cwd 및 상위 경로에서 탐색."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        d = p / 'data'
        if d.is_dir() and (d / 'train.csv').is_file():
            return str(d)
    raise FileNotFoundError(
        'data/train.csv 를 찾을 수 없습니다. '
        '터미널 cwd를 Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition 으로 두고 실행하세요.'
    )

path = _resolve_data_dir()
project_root = str(Path(path).resolve().parent)
print(f'▶ 데이터 경로: {path}')
print(f'▶ 제출 CSV 저장 위치(프로젝트 루트): {project_root}')

print('▶ 데이터 로드 중...', end=' ', flush=True)
t0 = time.time()
train = pd.read_csv(os.path.join(path, 'train.csv'))
test = pd.read_csv(os.path.join(path, 'test.csv'))
layout = pd.read_csv(os.path.join(path, 'layout_info.csv'))
print(f'완료 ({elapsed(t0)})  train {len(train):,}행 / test {len(test):,}행')

TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
N_FOLDS = 5
TARGET_LAG_COLS = ['target_lag1', 'target_lag2', 'target_lag3']


def handle_missing_values(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    cols_to_fix = [
        c for c in df.columns
        if df[c].isnull().any() and c not in (ID_COLS + [TARGET])
    ]
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].ffill()
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].bfill()
    df[cols_to_fix] = df[cols_to_fix].fillna(df[cols_to_fix].median())
    return df


def add_features(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    df['timeslot'] = df.groupby('scenario_id').cumcount()
    df['robot_efficiency'] = df['robot_active'] / (df['robot_total'] + 1e-6)
    df['robot_density'] = df['robot_total'] / (df['floor_area_sqm'] + 1e-6)
    df['order_per_station'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1e-6)
    df['robot_per_station'] = df['robot_active'] / (df['pack_station_count'] + 1e-6)
    df['cumulative_orders'] = df.groupby('scenario_id')['order_inflow_15m'].cumsum()
    df['order_pressure'] = df['cumulative_orders'] / (df['pack_station_count'] + 1e-6)
    if 'congestion_score' in df.columns:
        df['risk_index'] = df['congestion_score'] * (1 - df['robot_efficiency'])
        df['bottle_neck'] = df['order_per_station'] * df['congestion_score']
    if 'low_battery_ratio' in df.columns:
        df['battery_risk'] = df['low_battery_ratio'] * df['robot_total']
    if 'battery_mean' in df.columns and 'battery_std' in df.columns:
        df['battery_cv'] = df['battery_std'] / (df['battery_mean'] + 1e-6)
    if 'avg_trip_distance' in df.columns:
        df['trip_per_robot'] = df['avg_trip_distance'] / (df['robot_active'] + 1e-6)
        df['trip_congestion'] = df['avg_trip_distance'] * df.get('congestion_score', 0)
    if 'pack_utilization' in df.columns:
        df['pack_order_ratio'] = df['pack_utilization'] / (df['order_per_station'] + 1e-6)
    roll_cols = [
        'order_per_station', 'congestion_score', 'pack_utilization',
        'avg_trip_distance', 'low_battery_ratio',
    ]
    for col in roll_cols:
        if col not in df.columns:
            continue
        grp = df.groupby('scenario_id')[col]
        df[f'{col}_roll3_mean'] = grp.transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
        df[f'{col}_roll5_mean'] = grp.transform(
            lambda x: x.shift(1).rolling(5, min_periods=1).mean()
        )
        df[f'{col}_roll3_std'] = grp.transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0)
        )
    for lag in [1, 2]:
        if 'order_per_station' in df.columns:
            df[f'order_per_station_lag{lag}'] = (
                df.groupby('scenario_id')['order_per_station'].shift(lag).bfill()
            )
    scen_agg_cols = [
        'congestion_score', 'order_inflow_15m', 'battery_mean', 'pack_utilization',
        'avg_trip_distance', 'low_battery_ratio', 'max_zone_density', 'sku_concentration',
        'robot_idle', 'outbound_truck_wait_min', 'order_per_station', 'robot_efficiency',
        'order_pressure', 'risk_index', 'battery_risk', 'battery_cv',
    ]
    for col in scen_agg_cols:
        if col not in df.columns:
            continue
        stats = df.groupby('scenario_id')[col].agg(['mean', 'max', 'min', 'std']).reset_index()
        stats.columns = ['scenario_id'] + [f'{col}_scen_{f}' for f in ['mean', 'max', 'min', 'std']]
        df = df.merge(stats, on='scenario_id', how='left')
    if 'congestion_score_scen_mean' in df.columns and 'pack_utilization_scen_mean' in df.columns:
        df['cong_pack_interaction'] = (
            df['congestion_score_scen_mean'] * df['pack_utilization_scen_mean']
        )
    if 'avg_trip_distance_scen_mean' in df.columns and 'congestion_score_scen_mean' in df.columns:
        df['trip_cong_interaction'] = (
            df['avg_trip_distance_scen_mean'] * df['congestion_score_scen_mean']
        )
    if 'low_battery_ratio_scen_mean' in df.columns and 'robot_efficiency_scen_mean' in df.columns:
        df['battery_efficiency_risk'] = (
            df['low_battery_ratio_scen_mean'] * (1 - df['robot_efficiency_scen_mean'])
        )
    for col in ['congestion_score', 'order_per_station', 'pack_utilization', 'avg_trip_distance']:
        scen_mean_col = f'{col}_scen_mean'
        if col in df.columns and scen_mean_col in df.columns:
            df[f'{col}_rel_to_scen'] = df[col] / (df[scen_mean_col] + 1e-6)
    for col in ['congestion_score', 'order_per_station', 'pack_utilization']:
        if col not in df.columns:
            continue
        df[f'{col}_scen_rank'] = df.groupby('scenario_id')[col].rank(pct=True)
    return df


def preprocess_all(df, layout_df):
    df = df.merge(layout_df, on='layout_id', how='left')
    df = handle_missing_values(df)
    df = add_features(df)
    df['layout_type'] = pd.factorize(df['layout_type'])[0]
    return df


print('▶ 전처리 중...', end=' ', flush=True)
t0 = time.time()
train = preprocess_all(train, layout)
test = preprocess_all(test, layout)
print(f'완료 ({elapsed(t0)})')

# ============================================================
# 4. 타겟 인코딩 (원본과 동일)
# ============================================================
print('▶ 타겟 인코딩 중...', end=' ', flush=True)
t0 = time.time()
TE_COLS = [
    c for c in ['layout_id', 'timeslot', 'layout_type', 'shift_hour', 'day_of_week']
    if c in train.columns
]
SMOOTHING = 20
kf_te = GroupKFold(n_splits=N_FOLDS)
groups_te = train['scenario_id']
for col in TE_COLS:
    te_col = f'{col}_te'
    train[te_col] = np.nan
    global_mean = train[TARGET].mean()
    for tr_idx, val_idx in kf_te.split(train, train[TARGET], groups=groups_te):
        tr_df = train.iloc[tr_idx]
        stats = tr_df.groupby(col)[TARGET].agg(['mean', 'count'])
        smooth = (stats['count'] * stats['mean'] + SMOOTHING * global_mean) / (stats['count'] + SMOOTHING)
        train.loc[val_idx, te_col] = train.iloc[val_idx][col].map(smooth).fillna(global_mean)
    stats_full = train.groupby(col)[TARGET].agg(['mean', 'count'])
    smooth_full = (stats_full['count'] * stats_full['mean'] + SMOOTHING * global_mean) / (
        stats_full['count'] + SMOOTHING
    )
    test[te_col] = test[col].map(smooth_full).fillna(global_mean)
print(f'완료 ({elapsed(t0)})')

# 전역 타겟 lag는 넣지 않음 — fold 내부 / 테스트 순차에서만 생성
train_global_mean = train[TARGET].mean()
base_feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
feature_cols = base_feature_cols + TARGET_LAG_COLS
print(f'▶ base 피처 수: {len(base_feature_cols)} / +lag 후: {len(feature_cols)}')

for lag_col in TARGET_LAG_COLS:
    test[lag_col] = train_global_mean

y = np.log1p(train[TARGET])
groups = train['scenario_id']
kf = GroupKFold(n_splits=N_FOLDS)

oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
lgb_models, xgb_models, cat_models = [], [], []

# ============================================================
# 6. LightGBM
# ============================================================
section('LightGBM 학습 + 순차 OOF')
lgb_params = dict(
    n_estimators=10000,
    learning_rate=0.005,
    max_depth=8,
    num_leaves=127,
    min_child_samples=30,
    subsample=0.75,
    subsample_freq=1,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
lgb_fold_maes_seq = []
lgb_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    tr_raw = train.iloc[tr_idx].copy()
    va_raw = train.iloc[val_idx].copy()
    gm = tr_raw[TARGET].mean()

    tr_fit = add_target_lags_from_true_y(tr_raw)
    va_es = add_target_lags_from_true_y(va_raw)

    X_tr, y_tr = tr_fit[feature_cols], np.log1p(tr_fit[TARGET])
    X_es, y_es = va_es[feature_cols], np.log1p(va_es[TARGET])

    cb_p = LGBProgressCallback(fold, N_FOLDS, lgb_params['n_estimators'])
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_es, y_es)],
        eval_metric='mae',
        callbacks=[lgb.early_stopping(400, verbose=False), lgb.log_evaluation(-1), cb_p],
    )

    seq_pred = sequential_slot_predictions(
        model, va_raw, feature_cols, gm, lambda m, X: m.predict(X)
    )
    fold_idx = train.iloc[val_idx].index
    oof_lgb[val_idx] = seq_pred.reindex(fold_idx).to_numpy()

    lgb_models.append(model)
    fold_mae_seq = mean_absolute_error(train.loc[val_idx, TARGET], oof_lgb[val_idx])
    lgb_fold_maes_seq.append(fold_mae_seq)
    print(
        f"\n  ✔ Fold {fold}  OOF MAE(순차 lag) {fold_mae_seq:.4f}  "
        f"best iter {model.best_iteration_:,}  {elapsed(lgb_start)}"
    )

lgb_mae = mean_absolute_error(train[TARGET], oof_lgb)
print(f"\n  ▶ LightGBM OOF MAE (순차 lag): {lgb_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in lgb_fold_maes_seq]}")

# ============================================================
# 7. XGBoost
# ============================================================
section('XGBoost 학습 + 순차 OOF')
xgb_params = dict(
    n_estimators=10000,
    learning_rate=0.005,
    max_depth=7,
    subsample=0.75,
    colsample_bytree=0.6,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    tree_method='hist',
    early_stopping_rounds=400,
    eval_metric='mae',
    verbosity=0,
)
xgb_fold_maes_seq = []
xgb_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    tr_raw = train.iloc[tr_idx].copy()
    va_raw = train.iloc[val_idx].copy()
    gm = tr_raw[TARGET].mean()

    tr_fit = add_target_lags_from_true_y(tr_raw)
    va_es = add_target_lags_from_true_y(va_raw)

    X_tr, y_tr = tr_fit[feature_cols], np.log1p(tr_fit[TARGET])
    X_es, y_es = va_es[feature_cols], np.log1p(va_es[TARGET])

    cb_p = XGBProgressCallback(fold, N_FOLDS, 10000)
    model = xgb.XGBRegressor(**xgb_params, callbacks=[cb_p])
    model.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)

    seq_pred = sequential_slot_predictions(
        model, va_raw, feature_cols, gm, lambda m, X: m.predict(X)
    )
    fold_idx = train.iloc[val_idx].index
    oof_xgb[val_idx] = seq_pred.reindex(fold_idx).to_numpy()

    xgb_models.append(model)
    fold_mae_seq = mean_absolute_error(train.loc[val_idx, TARGET], oof_xgb[val_idx])
    xgb_fold_maes_seq.append(fold_mae_seq)
    print(
        f"\n  ✔ Fold {fold}  OOF MAE(순차 lag) {fold_mae_seq:.4f}  "
        f"best iter {model.best_iteration:,}  {elapsed(xgb_start)}"
    )

xgb_mae = mean_absolute_error(train[TARGET], oof_xgb)
print(f"\n  ▶ XGBoost OOF MAE (순차 lag): {xgb_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in xgb_fold_maes_seq]}")

# ============================================================
# 8. CatBoost
# ============================================================
section('CatBoost 학습 + 순차 OOF')
cat_params = dict(
    iterations=10000,
    learning_rate=0.005,
    depth=8,
    l2_leaf_reg=3.0,
    bootstrap_type='MVS',
    subsample=0.75,
    colsample_bylevel=0.6,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=42,
    task_type='CPU',
    early_stopping_rounds=400,
)
cat_fold_maes_seq = []
cat_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    tr_raw = train.iloc[tr_idx].copy()
    va_raw = train.iloc[val_idx].copy()
    gm = tr_raw[TARGET].mean()

    tr_fit = add_target_lags_from_true_y(tr_raw)
    va_es = add_target_lags_from_true_y(va_raw)

    X_tr, y_tr = tr_fit[feature_cols], np.log1p(tr_fit[TARGET])
    X_es, y_es = va_es[feature_cols], np.log1p(va_es[TARGET])

    print(f"\n  Fold {fold}/{N_FOLDS} 학습 중...")
    model = cb.CatBoostRegressor(**cat_params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=(X_es, y_es),
        verbose=cat_verbose_step(cat_params['iterations']),
        use_best_model=True,
    )

    seq_pred = sequential_slot_predictions(
        model, va_raw, feature_cols, gm, lambda m, X: m.predict(X)
    )
    fold_idx = train.iloc[val_idx].index
    oof_cat[val_idx] = seq_pred.reindex(fold_idx).to_numpy()

    cat_models.append(model)
    fold_mae_seq = mean_absolute_error(train.loc[val_idx, TARGET], oof_cat[val_idx])
    cat_fold_maes_seq.append(fold_mae_seq)
    print(
        f"  ✔ Fold {fold}  OOF MAE(순차 lag) {fold_mae_seq:.4f}  "
        f"best iter {model.best_iteration_:,}  {elapsed(cat_start)}"
    )

cat_mae = mean_absolute_error(train[TARGET], oof_cat)
print(f"\n  ▶ CatBoost OOF MAE (순차 lag): {cat_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in cat_fold_maes_seq]}")

# ============================================================
# 9. Test 순차 예측 (원본과 동일)
# ============================================================
section('Test 순차 예측 (배치)')

w_lgb = 1 / lgb_mae
w_xgb = 1 / xgb_mae
w_cat = 1 / cat_mae
w_sum = w_lgb + w_xgb + w_cat


def ensemble_predict(X, lgb_models, xgb_models, cat_models, w_lgb, w_xgb, w_cat, w_sum):
    p_lgb = np.mean([np.expm1(m.predict(X)) for m in lgb_models], axis=0)
    p_xgb = np.mean([np.expm1(m.predict(X)) for m in xgb_models], axis=0)
    p_cat = np.mean([np.expm1(m.predict(X)) for m in cat_models], axis=0)
    return (w_lgb * p_lgb + w_xgb * p_xgb + w_cat * p_cat) / w_sum


test = test.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
test['timeslot'] = test.groupby('scenario_id').cumcount()
max_slot = test['timeslot'].max()
test_preds = np.zeros(len(test))
seq_start = time.time()
print(f'  타임슬롯 0~{max_slot} 순차 예측 중...')

for slot in range(max_slot + 1):
    slot_mask = test['timeslot'] == slot
    slot_idx = test.index[slot_mask]
    for lag in [1, 2, 3]:
        lag_col = f'target_lag{lag}'
        prev_slot = slot - lag
        if prev_slot >= 0:
            prev_mask = test['timeslot'] == prev_slot
            prev_df = test.loc[prev_mask, ['scenario_id']].copy()
            prev_df['pred'] = test_preds[prev_mask]
            scen_to_pred = prev_df.set_index('scenario_id')['pred']
            scen_to_pred = scen_to_pred.groupby(level=0).last()
            test.loc[slot_idx, lag_col] = test.loc[slot_idx, 'scenario_id'].map(scen_to_pred).fillna(
                train_global_mean
            )
    X_slot = test.loc[slot_idx, feature_cols]
    test_preds[slot_idx] = ensemble_predict(
        X_slot, lgb_models, xgb_models, cat_models, w_lgb, w_xgb, w_cat, w_sum
    )
    if slot % 5 == 0 or slot == max_slot:
        print(f'  슬롯 {slot:>2}/{max_slot}  완료  {elapsed(seq_start)}', flush=True)

print(f'  ✔ 순차 예측 완료  ({elapsed(seq_start)})')

# ============================================================
# 10. 앙상블 & 제출
# ============================================================
section('앙상블 & 제출 (순차 OOF 가중치)')

oof_final = (w_lgb * oof_lgb + w_xgb * oof_xgb + w_cat * oof_cat) / w_sum
final_mae = mean_absolute_error(train[TARGET], oof_final)

print(f"  LightGBM {w_lgb/w_sum*100:.1f}%  (순차 OOF MAE {lgb_mae:.4f})")
print(f"  XGBoost  {w_xgb/w_sum*100:.1f}%  (순차 OOF MAE {xgb_mae:.4f})")
print(f"  CatBoost {w_cat/w_sum*100:.1f}%  (순차 OOF MAE {cat_mae:.4f})")
print(f"\n  ★ 앙상블 OOF MAE (순차 lag): {final_mae:.4f}")
print(f"  ★ 전체 소요 시간: {elapsed(lgb_start)}")

submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
save_path = os.path.join(project_root, 'submission_sequential_oof.csv')
submission.to_csv(save_path, index=False)
print(f"\n  저장 완료 → {save_path}")

▶ 데이터 경로: C:\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition\data
▶ 제출 CSV 저장 위치(프로젝트 루트): C:\Smart-Warehouse-Shipment-Delay-Prediction-AI-Competition
▶ 데이터 로드 중... 

완료 (0m 01s)  train 250,000행 / test 50,000행
▶ 전처리 중... 완료 (0m 30s)
▶ 타겟 인코딩 중... 완료 (0m 05s)
▶ base 피처 수: 214 / +lag 후: 217

  LightGBM 학습 + 순차 OOF
  Fold 1/5  [██████░░░░░░░░░░░░░░░░░░░░░░░░]  20.0%  iter  2000  val_MAE(진짜 lag, ES용) 0.2427  best 0.2427@2000  0m 52s
  ✔ Fold 1  OOF MAE(순차 lag) 11.4733  best iter 1,668  0m 57s
  Fold 2/5  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]  18.0%  iter  1800  val_MAE(진짜 lag, ES용) 0.2404  best 0.2403@1575  1m 13s
  ✔ Fold 2  OOF MAE(순차 lag) 11.3457  best iter 1,564  2m 21s
  Fold 3/5  [████░░░░░░░░░░░░░░░░░░░░░░░░░░]  16.0%  iter  1600  val_MAE(진짜 lag, ES용) 0.2459  best 0.2459@1597  1m 11s
  ✔ Fold 3  OOF MAE(순차 lag) 11.6757  best iter 1,369  3m 42s
  Fold 4/5  [██████░░░░░░░░░░░░░░░░░░░░░░░░]  22.0%  iter  2200  val_MAE(진짜 lag, ES용) 0.2377  best 0.2377@2002  1m 23s
  ✔ Fold 4  OOF MAE(순차 lag) 11.1962  best iter 1,830  5m 13s
  Fold 5/5  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]  18.0%  iter  1800  val_MAE(진짜 lag, ES용) 0.2402  best 0.2402@1704  1m 19s
  ✔ Fold 5  OO